# Notebook 05 — Speech Processing
## Speech Recognition & Synthesis

**Key questions:**
1. How does TTS for Indic languages differ from English TTS?
2. What does temperature control in Bulbul v3 actually sound like?
3. What is Word Error Rate (WER), and why is Indic ASR hard?
4. How does speech-to-text-translate work as a single-step pipeline?

## Theory: Speech Pipeline (covered in the speech processing chapters)

The ASR pipeline section describes the classical pipeline: **acoustic model** (maps audio frames to phoneme probabilities) → **language model** (scores token sequences) → **decoder** (beam search over combined scores).

Modern end-to-end systems (Whisper, Saaras) replace this with a single encoder-decoder transformer trained directly on (audio, transcript) pairs.

### Why Indic ASR is Harder

**1. Phoneme inventory size:**
- English: ~44 phonemes
- Hindi: ~50 phonemes (includes retroflexes: ट/ड vs त/द)
- Tamil: ~35 phonemes (but includes gemination: க vs க்க)
- Kannada/Telugu: ~56 phonemes (most complex Dravidian inventory)

**2. Retroflexion confusion:**
Hindi distinguishes dental `त` (ta) from retroflex `ट` (ṭa). Minimal pairs:
- `तन` (tan, body) vs `टन` (ṭan, ton)
This is acoustically subtle and confuses models trained primarily on English/European speech.

**3. Gemination in Tamil/Kannada:**
Tamil `கக்கா` (kakkā, crow) vs `காக்கா` — double consonants change meaning. Duration distinctions require fine-grained acoustic modeling.

**4. Code-mixed speech:**
A Hindi speaker will say: *'Meeting के बाद presentation submit करना है'*
The ASR must handle language switches mid-utterance.

### Saaras v3 Transcription Modes
| Mode | Output | Use case |
|------|--------|----------|
| transcribe | Native script | Standard transcription |
| verbatim | Native script + fillers | Verbatim/legal |
| translit | Roman transliteration | Keyboard-friendly |
| codemix | Mixed script output | Code-switched speech |

**Textbook Sections:** ASR pipeline overview, acoustic models, language models for ASR, text-to-speech synthesis, neural TTS

### Setup

Loads the API client, audio utilities, and visualization helpers for speech synthesis (TTS) and recognition (STT) experiments.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from utils.sarvam_helpers import (
    load_client, reset_demo_counters, tts_audio,
    display_audio_inline, plot_waveform
)
from data.sample_texts import SAMPLE_TEXTS, LANGUAGE_NAMES
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

reset_demo_counters()
client = load_client()
os.makedirs('../outputs/audio', exist_ok=True)
print('Ready')

### TTS demo: generate audio for all 4 sample sentences

We use Sarvam's **Bulbul v3** TTS model to synthesize speech for our four canonical texts — hearing the same content in Hindi, Tamil, Bengali, and Telugu.


<details>
<summary><strong>How does Text-to-Speech (TTS) work?</strong></summary>

Modern TTS systems have three stages:

**1. Text analysis** (front-end):
- Normalize text: '₹500' → 'पाँच सौ रुपये'
- Grapheme-to-phoneme (G2P): Convert script to phoneme sequence
- Prosody prediction: Where do pauses, stress, and intonation go?

**2. Acoustic model** (back-end):
- Convert phoneme sequence to mel spectrogram (a visual representation of audio frequencies over time)
- Modern models: Tacotron 2, FastSpeech 2, VITS (variational inference TTS)

**3. Vocoder:**
- Convert mel spectrogram to actual audio waveform
- WaveNet (slow but high quality), HiFi-GAN (fast, near-WaveNet quality)

**Indic TTS challenges:**
- **Schwa deletion**: Hindi silently drops the inherent 'a' vowel in many positions — 'राम' is 'Raam' not 'Raama'. Rules differ by dialect.
- **Retroflex consonants**: ट, ड, ण require tongue placement that doesn't exist in English — the acoustic model must produce distinct sounds.
- **Gemination**: Doubled consonants in Tamil/Kannada (e.g., 'அம்மா' = ammā) must be audibly longer.
</details>


In [ ]:
reset_demo_counters()

print('TTS DEMO — Bulbul v3')
print('Generating audio for all 4 sample texts...')
print('='*60)

# Use shorter excerpts for demo
demo_texts = {
    'hi-IN': 'विद्यालय में शिक्षक छात्रों को भाषा समझा रहे हैं।',
    'ta-IN': 'மாணவர்கள் கணினி அறிவியல் படிக்கிறார்கள்.',
    'bn-IN': 'আজকাল ভাষা প্রক্রিয়াকরণ অনেক প্রযুক্তিতে ব্যবহৃত হচ্ছে।',
    'te-IN': 'భాషా ప్రాసెసింగ్ చాలా ముఖ్యమైన పాత్ర పోషిస్తుంది.',
}

# bulbul:v3 supported languages
v3_langs = {'hi-IN', 'ta-IN', 'bn-IN', 'te-IN', 'kn-IN', 'ml-IN', 'mr-IN', 'gu-IN', 'pa-IN', 'od-IN'}

for lang, text in demo_texts.items():
    try:
        audio = tts_audio(client, text, lang=lang, voice='ritu', model='bulbul:v3')
        audio_path = f'../outputs/audio/demo_{lang}.wav'
        with open(audio_path, 'wb') as f:
            f.write(audio)
        print(f'[{LANGUAGE_NAMES[lang]}] {text[:40]}...')
        print(f'  Saved: {audio_path} ({len(audio)} bytes)')
        display_audio_inline(audio, label=f'{LANGUAGE_NAMES[lang]} TTS')
    except Exception as e:
        print(f'[{LANGUAGE_NAMES[lang]}] Error: {e}')

### Voice comparison: 3 different voices on same Hindi text

Same text, different voices — this demonstrates the **multi-speaker** capability of modern TTS models, where speaker identity is disentangled from linguistic content.


<details>
<summary><strong>Multi-speaker TTS: how one model produces different voices</strong></summary>

Modern TTS models separate **what is said** from **how it's said**:

- **Content encoder**: Processes the text (phonemes, prosody)
- **Speaker embedding**: A learned vector (typically 256-dim) that captures vocal characteristics — pitch range, timbre, speaking rate
- **Decoder**: Combines content + speaker embedding to generate speaker-specific audio

**Training:** The model is trained on data from multiple speakers. Each speaker gets an embedding vector. At inference, you select a speaker embedding to control the voice.

**Sarvam's Bulbul v3** offers voices like 'meera', 'ritu', 'amol' — each with distinct vocal characteristics. The underlying linguistic content (pronunciation, phoneme durations) stays the same.
</details>


In [ ]:
reset_demo_counters()

hindi_text = 'नमस्ते, मैं आपका स्वागत करता हूँ। आज हम भाषा विज्ञान के बारे में सीखेंगे।'

# Bulbul v3 available voices
voices_v3 = ['ritu', 'shubh', 'arya']

print('VOICE COMPARISON — Bulbul v3 (Hindi)')
print(f'Text: {hindi_text}')
print('='*60)

for voice in voices_v3:
    try:
        audio = tts_audio(client, hindi_text, lang='hi-IN', voice=voice, model='bulbul:v3')
        audio_path = f'../outputs/audio/voice_{voice}.wav'
        with open(audio_path, 'wb') as f:
            f.write(audio)
        print(f'Voice: {voice} ({len(audio)} bytes)')
        display_audio_inline(audio, label=f'Voice: {voice}')
    except Exception as e:
        print(f'Voice {voice}: Error — {e}')

### Temperature effect demo: flat vs default vs expressive

**Temperature** in TTS controls expressiveness — lower values produce flat, monotone speech while higher values produce animated, expressive delivery. We demo the range on the same sentence.


<details>
<summary><strong>What does 'temperature' mean in neural models?</strong></summary>

Temperature is a parameter that controls randomness in neural model outputs:

**In language models (LLMs):** Temperature scales the logits before softmax.
- T=0: Always pick the most likely token (deterministic, repetitive)
- T=1: Sample from the learned distribution (balanced)
- T>1: Flatten the distribution (more random, creative)

**In TTS:** Temperature controls the variance in acoustic model predictions:
- Low T (0.3): Minimal variation in pitch/energy → flat, robotic delivery
- Default T (0.7): Natural-sounding prosody
- High T (1.0+): Exaggerated pitch contours, more expressive → can sound theatrical or unstable

**The tradeoff:** Higher temperature = more natural/expressive but also more likely to produce artifacts (pitch jumps, mispronunciations). Production systems typically use T=0.7–0.8 as a sweet spot.
</details>


In [ ]:
reset_demo_counters()

text = 'आज का दिन बहुत अच्छा है! हम सब मिलकर कुछ नया सीखेंगे।'
temperatures = [0.1, 0.6, 1.4]
labels = ['0.1 (flat/monotone)', '0.6 (default/natural)', '1.4 (expressive/dramatic)']

print('TEMPERATURE EFFECT — Bulbul v3')
print(f'Text: {text}')
print('='*60)
print('Note: Higher temperature = more prosodic variation in pitch and rhythm')
print()

for temp, label in zip(temperatures, labels):
    try:
        audio = tts_audio(client, text, lang='hi-IN', voice='ritu', model='bulbul:v3', temperature=temp)
        audio_path = f'../outputs/audio/temp_{str(temp).replace(".","_")}.wav'
        with open(audio_path, 'wb') as f:
            f.write(audio)
        print(f'Temperature {label}: {len(audio)} bytes')
        display_audio_inline(audio, label=f'Temperature {label}')
    except Exception as e:
        print(f'Temperature {temp}: Error — {e}')

### TTS waveform visualization

Visualizing the raw audio waveform and mel spectrogram of synthesized speech. The waveform shows amplitude over time; the spectrogram shows frequency content — together they reveal prosody, pauses, and speaking rate.


<details>
<summary><strong>Reading waveforms and spectrograms</strong></summary>

**Waveform** (time domain): Amplitude (loudness) vs time.
- Tall peaks = loud sounds (vowels, stressed syllables)
- Flat regions = silence/pauses
- Rapid oscillation = high-frequency sounds (sibilants like 's', 'sh')

**Mel spectrogram** (frequency domain): A 2D image where:
- X-axis = time
- Y-axis = frequency (mel scale — perceptually uniform, compressed at high frequencies)
- Color intensity = energy at that frequency and time

**What to look for:**
- **Formant bands**: Horizontal bright bands = vowel formants (F1, F2). Different vowels have different formant patterns.
- **Pitch contour**: The fundamental frequency (F0) visible as the lowest periodic band. Rising = question, falling = statement.
- **Silences**: Gaps between words. Indian languages typically have shorter inter-word pauses than English.
</details>


In [ ]:
reset_demo_counters()

text = 'नमस्ते भारत।'
try:
    audio = tts_audio(client, text, lang='hi-IN', voice='ritu', model='bulbul:v3', temperature=0.6)
    print(f'Generated audio: {len(audio)} bytes')
    plot_waveform(audio, sample_rate=22050, title=f'TTS Waveform — Bulbul v3\n"{text}" (Hindi, voice=ritu, temp=0.6)')
except Exception as e:
    print(f'Error: {e}')

## Theory: Word Error Rate (WER) — The Indic ASR Challenge

The acoustic models section defines **Word Error Rate** as:
```
WER = (Substitutions + Deletions + Insertions) / Total Reference Words
```

Lower is better. Published benchmarks for Indic languages (Vistaar benchmark, 2023):

| System | Hindi WER | Tamil WER | Bengali WER | Telugu WER |
|--------|-----------|-----------|-------------|------------|
| Saaras v3 (Sarvam) | ~19% | ~25% | ~22% | ~28% |
| Whisper Large v3 | ~15% | ~31% | ~20% | ~35% |
| MMS (Meta) | ~23% | ~28% | ~25% | ~30% |
| IndicWav2Vec (AI4Bharat) | ~20% | ~26% | ~21% | ~27% |

**Key observations:**
1. Whisper wins on Hindi (large multilingual training), but struggles on Dravidian (Tamil, Telugu)
2. Saaras v3 is competitive across all 4 languages — more balanced coverage
3. WER degrades for longer sentences with code-switching
4. Numbers and proper nouns are major error sources in all systems

### WER comparison bar chart

Comparing Word Error Rate across different ASR systems and languages — this visualization shows where each system excels and where it struggles.


<details>
<summary><strong>Understanding Word Error Rate (WER)</strong></summary>

**WER** = (Substitutions + Insertions + Deletions) / Total words in reference

WER is computed by finding the minimum edit distance between the ASR output and the reference transcript.

**Interpretation:**
- WER 5-10%: Excellent (human-level for clear speech)
- WER 10-20%: Good (usable for most applications)
- WER 20-40%: Mediocre (needs post-processing or human review)
- WER > 40%: Poor (unreliable)

**WER can exceed 100%** if the model hallucinates many extra words (insertions > total reference words).

**Indic ASR difficulty ranking** (typical):
- Hindi: easiest (most training data, 50-100K hours)
- Bengali/Tamil: moderate (10-50K hours)
- Telugu/Kannada: harder (less data, complex phonology)
- Northeastern languages (Assamese, Manipuri): hardest (< 1K hours)
</details>


In [ ]:
reset_demo_counters()

# Published benchmark numbers (Vistaar benchmark, approximate)
wer_data = {
    'Saaras v3 (Sarvam)':    {'hi-IN': 19, 'ta-IN': 25, 'bn-IN': 22, 'te-IN': 28, 'mni-IN': 35, 'pa-IN': 24, 'kok-IN': 32},
    'Whisper Large v3':       {'hi-IN': 15, 'ta-IN': 31, 'bn-IN': 20, 'te-IN': 35, 'mni-IN': 45, 'pa-IN': 22, 'kok-IN': 40},
    'MMS (Meta)':             {'hi-IN': 23, 'ta-IN': 28, 'bn-IN': 25, 'te-IN': 30, 'mni-IN': 40, 'pa-IN': 27, 'kok-IN': 38},
    'IndicWav2Vec':           {'hi-IN': 20, 'ta-IN': 26, 'bn-IN': 21, 'te-IN': 27, 'mni-IN': 38, 'pa-IN': 23, 'kok-IN': 34},
}

systems = list(wer_data.keys())
languages = ['hi-IN', 'ta-IN', 'bn-IN', 'te-IN', 'mni-IN', 'pa-IN', 'kok-IN']
lang_names = ['Hindi', 'Tamil', 'Bengali', 'Telugu', 'Meitei', 'Punjabi', 'Konkani']
x = np.arange(len(lang_names))
width = 0.2
colors = ['#FF6B35', '#4472C4', '#70AD47', '#FFC000']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (system, data) in enumerate(wer_data.items()):
    wers = [data[l] for l in languages]
    offset = (i - len(systems)/2 + 0.5) * width
    bars = ax.bar(x + offset, wers, width, label=system, color=colors[i], alpha=0.85, edgecolor='white')
    ax.bar_label(bars, fmt='%d%%', padding=2, fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(lang_names)
ax.set_ylabel('Word Error Rate (%) — Lower is Better')
ax.set_title('ASR Word Error Rate Comparison — Indic Languages\n(Vistaar benchmark, approximate published values)')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 55)
sns.despine()
plt.tight_layout()
plt.savefig('../outputs/figures/05_wer_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Key Takeaways (Speech Recognition & Synthesis)

1. **Bulbul v3's temperature parameter is prosodic control.** Unlike volume/pitch (v2), temperature modulates the *naturalness* of speech variation — higher values produce more human-like prosodic patterns but less consistency.

2. **Voice diversity in TTS is not just aesthetic.** Different voices capture different gender, age, and regional accent patterns. For Indian users, voice familiarity significantly affects user acceptance.

3. **WER is script-dependent.** Tamil's long agglutinated words count as single word errors even if most of the morpheme string is correct. WER systematically underestimates quality for Tamil/Telugu.

4. **Whisper ≠ best for all Indic languages.** Despite strong Hindi performance (15% WER), Whisper Large v3 struggles on Tamil (31%) because it saw far less Tamil training data. Domain-specific models like Saaras trade Hindi accuracy for Tamil/Telugu balance.

5. **End-to-end speech translation eliminates error cascading.** Two-step STT→MT propagates ASR errors into translation. Direct speech translation (what Saaras translate mode offers) can be more robust.

**Next:** Notebook 06 — quantitative model comparisons and benchmark analysis.

---
## ⭐ Bonus — TTS → STT Round-Trip Fidelity Test

> **Skip if time is short.** This section chains Bulbul v3 (TTS) directly into
> Saaras v3 (STT) to measure how much text is lost or distorted in the full
> speech loop. It is the closest thing to an end-to-end integration test.

### Background
The speech processing chapters treat TTS and ASR as separate systems. In
production, they are often chained:
```
Text → [Bulbul TTS] → Audio → [Saaras STT] → Transcribed text
```
The gap between input text and transcribed text is the **round-trip error**.
Contributing factors:
- TTS prosody artefacts (unusual pitch contours confuse the acoustic model)
- Vocabulary mismatch (TTS may produce clear audio of a rare word that ASR
  has never seen in training)
- Sampling rate conversion losses
- Conjunct character rendering differences (Bengali, Hindi)

We measure round-trip quality with **Character Error Rate (CER)** rather than
WER because:
- CER is fairer to agglutinative languages (one Tamil word ≠ one English word)
- Script-level errors (wrong Devanagari character) are visible at CER level
- CER = (edit distance at character level) / (reference length)


In [ ]:
# ⭐ BONUS — TTS → STT round-trip fidelity (Bulbul v3 → Saaras v3)
# Measures Character Error Rate (CER) for the full speech loop
import sys, os, tempfile, wave, struct
sys.path.insert(0, os.path.abspath('..'))
from utils.sarvam_helpers import load_client, reset_demo_counters, tts_audio

reset_demo_counters()
client = load_client()

# ── CER helper ────────────────────────────────────────────────────────────
def char_error_rate(reference: str, hypothesis: str) -> float:
    """Levenshtein edit distance at character level, normalised by ref length."""
    ref = list(reference.replace(" ", ""))
    hyp = list(hypothesis.replace(" ", ""))
    if not ref:
        return 0.0
    # Dynamic programming
    dp = list(range(len(hyp) + 1))
    for i, rc in enumerate(ref):
        new_dp = [i + 1]
        for j, hc in enumerate(hyp):
            new_dp.append(min(
                dp[j + 1] + 1,          # deletion
                new_dp[j]  + 1,          # insertion
                dp[j] + (0 if rc == hc else 1),  # substitution
            ))
        dp = new_dp
    return dp[-1] / len(ref)

# ── Test sentences ─────────────────────────────────────────────────────────
test_cases = [
    {"lang": "hi-IN", "text": "शिक्षक छात्रों को भाषा विज्ञान पढ़ा रहे हैं।"},
    {"lang": "ta-IN", "text": "மாணவர்கள் கணினி அறிவியல் படிக்கிறார்கள்."},
    {"lang": "bn-IN", "text": "আজকাল ভাষা প্রক্রিয়াকরণ অনেক প্রযুক্তিতে ব্যবহৃত হচ্ছে।"},
]

LANG_NAMES = {"hi-IN": "Hindi", "ta-IN": "Tamil", "bn-IN": "Bengali"}

print("TTS → STT ROUND-TRIP FIDELITY TEST")
print("="*65)
print(f"{'Language':<10} {'CER':>6}  {'Input text (truncated)'}")
print("-"*65)

cer_results = {}
for tc in test_cases:
    lang, text = tc["lang"], tc["text"]
    try:
        # Step 1: TTS — text → audio bytes
        audio_bytes = tts_audio(client, text, lang=lang,
                                voice="ritu", model="bulbul:v3",
                                temperature=0.6)

        # Step 2: Save to temp WAV file (Saaras expects a file)
        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        tmp.write(audio_bytes)
        tmp.close()

        # Step 3: STT — audio → transcribed text
        with open(tmp.name, "rb") as f:
            stt_response = client.speech_to_text.transcribe(
                file=("audio.wav", f, "audio/wav"),
                language_code=lang,
                model="saaras:v2",
                with_timestamps=False,
            )
        os.unlink(tmp.name)

        transcript = getattr(stt_response, "transcript", "") or ""
        if not transcript:
            # try alternative attribute names
            transcript = (getattr(stt_response, "text", "") or
                          str(stt_response))

        cer = char_error_rate(text, transcript)
        cer_results[lang] = cer
        print(f"{LANG_NAMES[lang]:<10} {cer:>5.1%}  {text[:40]}...")
        print(f"           transcript: {transcript[:60]}")

    except Exception as e:
        cer_results[lang] = None
        print(f"{LANG_NAMES[lang]:<10}  Error: {str(e)[:70]}")

# ── Visualization ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt, seaborn as sns
valid = {k: v for k, v in cer_results.items() if v is not None}
if valid:
    langs  = [LANG_NAMES[l] for l in valid]
    cers   = [v * 100 for v in valid.values()]
    colors = ["#27AE60" if c < 15 else "#E67E22" if c < 30 else "#E74C3C"
              for c in cers]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.bar(langs, cers, color=colors, edgecolor="white", width=0.5)
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
    ax.axhline(15, color="#27AE60", linewidth=1.2, linestyle="--",
               label="15% — good CER threshold")
    ax.set_ylabel("Character Error Rate (%)")
    ax.set_title("TTS → STT Round-Trip CER\n"
                 "Bulbul v3 → Saaras v2  |  Lower is better")
    ax.legend(fontsize=9)
    sns.despine()
    plt.tight_layout()
    plt.savefig("../outputs/figures/05_bonus_roundtrip_cer.png",
                dpi=120, bbox_inches="tight")
    plt.show()

print("\nInterpretation:")
print("  CER < 15% : Excellent — pipeline is production-ready")
print("  CER 15-30%: Acceptable — occasional character errors in complex words")
print("  CER > 30% : Poor — likely an OOV / script rendering issue")
print("\n  Tamil typically shows higher CER because agglutinated forms")
print("  that TTS renders correctly may not appear in Saaras training data")
print("  as a single unit — causing split-word transcription errors.")


---
## ⭐ Bonus — Failure Mode: Long Sentence Prosody Degradation

> **Skip if time is short.** This cell demonstrates how Bulbul v3 handles
> sentences at the edge of its designed operating range, and what degrades.

### Background
The neural TTS section describes how sequence-to-sequence TTS models produce
mel-spectrograms that are then converted to waveforms. Like all seq2seq models,
performance degrades as input length grows:
- **Short sentences (< 50 chars):** Prosody is natural, pitch varies appropriately
- **Medium sentences (50–150 chars):** Slight flattening at clause boundaries
- **Long sentences (> 200 chars):** Common failure modes appear:
  - Monotone stretches in the middle (attention drifts)
  - Repeated words or phrases (attention loops)
  - Truncation (model hits max output length)
  - Unnatural pauses at non-boundary positions

We test this systematically by generating the same content at three lengths
and comparing waveform energy profiles.


In [ ]:
# ⭐ BONUS — Failure mode: TTS prosody degradation at long sentence lengths
import sys, os, struct, wave, io
sys.path.insert(0, os.path.abspath('..'))
from utils.sarvam_helpers import load_client, reset_demo_counters, tts_audio, plot_waveform

reset_demo_counters()
client = load_client()

sentences = {
    "Short  (45 chars)" : "शिक्षक छात्रों को भाषा समझा रहे हैं।",
    "Medium (95 chars)" : ("शिक्षक आज कक्षा में छात्रों को प्राकृतिक भाषा प्रसंस्करण "
                           "और मशीन लर्निंग के बारे में विस्तार से समझा रहे हैं।"),
    "Long  (195 chars)" : ("आज की कक्षा में शिक्षक जी ने छात्रों को बताया कि प्राकृतिक "
                           "भाषा प्रसंस्करण एक ऐसा क्षेत्र है जो भाषा विज्ञान, कंप्यूटर "
                           "विज्ञान और कृत्रिम बुद्धिमत्ता को मिलाकर कंप्यूटर को मानव "
                           "भाषा समझने और उत्पन्न करने में सक्षम बनाता है।"),
}

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

fig, axes = plt.subplots(len(sentences), 1, figsize=(13, 3 * len(sentences)))
if len(sentences) == 1:
    axes = [axes]

for ax, (label, text) in zip(axes, sentences.items()):
    char_count = len(text)
    print(f"Generating: {label}  ({char_count} chars)")
    try:
        audio_bytes = tts_audio(client, text, lang="hi-IN",
                                voice="ritu", model="bulbul:v3",
                                temperature=0.6)
        # Parse WAV
        with wave.open(io.BytesIO(audio_bytes)) as wf:
            n_frames  = wf.getnframes()
            n_chan    = wf.getnchannels()
            sw        = wf.getsampwidth()
            sr        = wf.getframerate()
            raw       = wf.readframes(n_frames)
        fmt     = {1: "b", 2: "h", 4: "i"}.get(sw, "h")
        samples = np.array(struct.unpack(f"<{n_frames * n_chan}{fmt}", raw),
                           dtype=np.float32)
        if n_chan > 1:
            samples = samples[::n_chan]
        t = np.linspace(0, len(samples) / sr, len(samples))

        # RMS energy in 50 ms windows (to see prosodic rhythm)
        win   = int(0.05 * sr)
        rms   = [np.sqrt(np.mean(samples[i:i+win]**2))
                 for i in range(0, len(samples)-win, win//2)]
        t_rms = np.linspace(0, len(samples)/sr, len(rms))

        ax.plot(t, samples / (np.abs(samples).max() + 1e-9),
                color="#cccccc", linewidth=0.3, alpha=0.7, label="waveform")
        ax.plot(t_rms, np.array(rms) / (max(rms) + 1e-9),
                color="#FF6B35", linewidth=1.5, label="RMS energy envelope")
        ax.set_title(f"{label}  |  {len(audio_bytes)//1024} KB  |  {len(samples)/sr:.1f}s",
                     fontsize=10)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude")
        ax.legend(fontsize=8, loc="upper right")
        print(f"  → {len(audio_bytes)//1024} KB,  {len(samples)/sr:.1f}s audio")

    except Exception as e:
        ax.set_title(f"{label} — Error: {e}")
        print(f"  Error: {e}")

plt.suptitle("TTS Waveform Comparison by Sentence Length (Hindi, Bulbul v3)\n"
             "Watch for flat RMS envelope in the long sentence = prosody degradation",
             fontsize=11, y=1.01)
sns.despine()
plt.tight_layout()
plt.savefig("../outputs/figures/05_bonus_prosody_length.png",
            dpi=120, bbox_inches="tight")
plt.show()

print("\nWhat to observe:")
print("  Short   — RMS envelope has clear peaks and valleys (natural rhythm)")
print("  Medium  — Similar rhythm, slight flattening near clause boundary")
print("  Long    — Watch for a monotone flat stretch in the middle third:")
print("            this is the attention drift zone where the model loses")
print("            track of prosodic structure over the long context window.")


---
## ⭐ Bonus: TTS → STT Round-Trip Pipeline
> **Skip if short on time.** This cell chains Bulbul v3 (TTS) directly into Saaras v3 (STT) to create a complete speech round-trip. The output quantifies how much text is preserved through the audio channel — a direct measure of combined TTS+STT system fidelity.

### Why a round-trip matters
In a voice assistant pipeline:
```
Text input → TTS (Bulbul) → Audio → ASR (Saaras) → Text output
                                          ↑
                                   errors accumulate here
```

TTS errors (mispronunciation, prosody issues) compound with ASR errors (wrong phoneme recognition). A round-trip test reveals the *combined* system fidelity without needing a human to speak into a microphone.

### Measuring round-trip fidelity
We use **Character Error Rate (CER)** instead of WER here, because:
- Indic scripts have complex grapheme clusters (Devanagari matras, Tamil uyir-mei)
- A single ASR substitution may differ by just one character (dental vs retroflex)
- CER is more granular and better captures partial correctness

```
CER = (char_substitutions + char_deletions + char_insertions) / len(reference)
```

**Expected results:** Well-pronounced common words → CER < 10%. Loan words, technical terms, numerals → CER can exceed 40% even on a clean pipeline.

### What Saaras v3 modes return
| Mode | What it outputs | When to use |
|------|----------------|-------------|
| `transcribe` | Native script (Devanagari for Hindi) | Standard pipeline |
| `verbatim` | Native script + fillers (um, uh) | Legal/medical transcription |
| `translit` | Roman transliteration | Keyboard input, search |
| `codemix` | Mixed EN+Indic | Code-switched speech |

### ⭐ BONUS — TTS → STT Round-Trip with CER Measurement

In [ ]:
# Uses: Bulbul v3 (TTS) → WAV file → Saaras v3 (STT) → compare with original text
# No extra packages needed beyond the core requirements

import sys, os, io, wave, struct, base64
sys.path.insert(0, os.path.abspath('..'))

from utils.sarvam_helpers import load_client, reset_demo_counters, tts_audio, display_audio_inline
from data.sample_texts import SAMPLE_TEXTS, LANGUAGE_NAMES
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

reset_demo_counters()
client = load_client()
os.makedirs('../outputs/audio', exist_ok=True)

# ── CER implementation ─────────────────────────────────────────────────────
def char_error_rate(reference: str, hypothesis: str) -> float:
    """Compute Character Error Rate using dynamic programming (edit distance)."""
    ref = list(reference.strip())
    hyp = list(hypothesis.strip())
    if not ref:
        return 1.0
    # Wagner-Fischer edit distance
    d = np.zeros((len(ref) + 1, len(hyp) + 1), dtype=int)
    for i in range(len(ref) + 1):
        d[i, 0] = i
    for j in range(len(hyp) + 1):
        d[0, j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i, j] = min(d[i-1, j] + 1,      # deletion
                         d[i, j-1] + 1,         # insertion
                         d[i-1, j-1] + cost)     # substitution
    return d[len(ref), len(hyp)] / len(ref)

# ── Test sentences (short, clear, common vocabulary) ──────────────────────
test_cases = [
    ("hi-IN", "नमस्ते, आज मौसम बहुत अच्छा है।",          "ritu"),
    ("hi-IN", "भाषा प्रसंस्करण एक महत्वपूर्ण विषय है।",   "ritu"),
    ("ta-IN", "வணக்கம், இன்று வானிலை நன்றாக உள்ளது.",     "ritu"),
]

print("TTS → STT ROUND-TRIP FIDELITY TEST")
print("="*65)
print(f"{'Language':<10} {'CER':>6}  {'Input text (truncated)'}")
print("-"*65)

results = []
for lang, text, voice in test_cases:
    lang_name = LANGUAGE_NAMES[lang]
    print(f"\n[{lang_name}]")
    print(f"  Input:  {text}")

    # Step 1: TTS — generate audio
    try:
        audio_bytes = tts_audio(client, text, lang=lang, voice=voice,
                                model="bulbul:v3", temperature=0.5)
        audio_path = f"../outputs/audio/roundtrip_{lang.replace('-','_')}.wav"
        with open(audio_path, "wb") as f:
            f.write(audio_bytes)
        print(f"  TTS:    {len(audio_bytes):,} bytes written to {audio_path}")
        display_audio_inline(audio_bytes, label=f"Generated audio ({lang_name})")
    except Exception as e:
        print(f"  TTS Error: {e}")
        results.append({"lang": lang_name, "input": text, "output": "TTS_ERROR", "cer": 1.0})
        continue

    # Step 2: STT — transcribe back
    transcription = None
    try:
        with open(audio_path, "rb") as f:
            audio_file = f.read()

        # Try Saaras v3 speech-to-text
        stt_response = client.speech_to_text.transcribe(
            file=("audio.wav", audio_file, "audio/wav"),
            language_code=lang,
            model="saaras:v2",
        )
        if hasattr(stt_response, "transcript"):
            transcription = stt_response.transcript
        elif hasattr(stt_response, "text"):
            transcription = stt_response.text
        else:
            transcription = str(stt_response)
        print(f"  STT:    {transcription}")
    except Exception as e:
        print(f"  STT Error: {e}")
        print("  (STT requires audio file support — check Saaras API access)")
        transcription = None

    if transcription:
        cer = char_error_rate(text, transcription)
        print(f"  CER:    {cer:.1%}")
        results.append({"lang": lang_name, "input": text,
                        "output": transcription, "cer": cer})
    else:
        results.append({"lang": lang_name, "input": text,
                        "output": "STT_UNAVAILABLE", "cer": None})

# ── Visualise CER results ──────────────────────────────────────────────────
valid_results = [r for r in results if r["cer"] is not None]
if valid_results:
    langs_plot = [r["lang"] for r in valid_results]
    cers_plot  = [r["cer"] * 100 for r in valid_results]
    colors     = ["#27AE60" if c < 15 else "#F39C12" if c < 35 else "#E74C3C"
                  for c in cers_plot]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(langs_plot, cers_plot, color=colors, edgecolor="white", linewidth=1.5)
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=11)
    ax.axhline(10, color="#27AE60", linestyle="--", linewidth=1, label="Excellent (< 10%)")
    ax.axhline(30, color="#E74C3C", linestyle="--", linewidth=1, label="Poor (> 30%)")
    ax.set_ylabel("Character Error Rate (%)")
    ax.set_title("TTS → STT Round-Trip Character Error Rate\n"
                "(Bulbul v3 → Saaras v3; lower = better)")
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(cers_plot) * 1.3 + 5)
    sns.despine()
    plt.tight_layout()
    plt.savefig("../outputs/figures/05_bonus_roundtrip_cer.png", dpi=120, bbox_inches="tight")
    plt.show()

    print("\n── Round-Trip Summary ───────────────────────────────────────")
    print("CER < 10%  → Excellent fidelity (native script, common vocabulary)")
    print("CER 10-30% → Acceptable (minor substitutions, usually readable)")
    print("CER > 30%  → Degraded (loan words, technical terms, or STT failure)")
    avg_cer = np.mean([r["cer"] for r in valid_results])
    print(f"\nOverall average CER: {avg_cer:.1%}")
else:
    print("\nNo STT results to visualise.")
    print("If STT returned errors, it may require a different audio format or API access level.")
    print("The TTS step (Bulbul v3) succeeded — play the audio cells above to verify output quality.")